[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HunterBushnell/SCP/blob/main/0_pipeline.ipynb)

# SCP Pipeline - Compact Steps 1-5

This is the simplest end-to-end SCP notebook for an already staged cell. It fills missing configs safely, builds one cell for passive/active/synapse tuning, then runs the final simulation in a clean process and displays its result.

**How to use it:** choose **Run All** once to render the five small step panels, then click each enabled step in order. Run All does not start a model or simulation by itself.

Use `1_setup.ipynb` for full model import/setup control. Use the numbered notebooks when you need ACT active tuning, target-source setup, exports, synapse optimization, or other advanced options.

> **Manual tuning rule:** after editing a fit JSON, HOC/morphology source, `cell_config.json`, or MOD source, restart the kernel and rerun from the top. Runtime, target, geometry, and synapse config edits are reloaded where they are used.

## Notebook Setup: Environment

Run this once. The cell finds SCP locally or clones it in a fresh Colab runtime. Its source is collapsed to keep the workflow focused.

In [ ]:
try:
    ip = get_ipython()
except NameError:
    ip = None
if ip is not None:
    try:
        ip.run_line_magic("load_ext", "autoreload")
        ip.run_line_magic("autoreload", "2")
    except Exception as exc:
        print(f"IPython autoreload unavailable ({exc}); continuing.")

import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
SCP_REPO_URL = os.environ.get("SCP_REPO_URL", "https://github.com/HunterBushnell/SCP.git")
SCP_REPO_BRANCH = os.environ.get("SCP_REPO_BRANCH", "") or None
SCP_REPO_DIR = Path(os.environ.get("SCP_REPO_DIR", "/content/SCP" if IN_COLAB else str(Path.cwd() / "SCP")))
INSTALL_DEPS = None  # None installs automatically in Colab and not locally.

def _looks_like_scp(path):
    return (path / "modules").is_dir() and (path / "run_pipeline.py").is_file()

repo_root = None
env_root = os.environ.get("SCP_ROOT")
if env_root and _looks_like_scp(Path(env_root).expanduser()):
    repo_root = Path(env_root).expanduser().resolve()
else:
    start = Path.cwd().resolve()
    candidates = [start, *start.parents]
    for base in (start, start.parent):
        try:
            candidates.extend(child for child in base.iterdir() if child.is_dir())
        except Exception:
            pass
    for candidate in candidates:
        if _looks_like_scp(candidate):
            repo_root = candidate.resolve()
            break

if repo_root is None:
    if not IN_COLAB and os.environ.get("SCP_AUTO_CLONE", "0") not in {"1", "true", "True"}:
        raise FileNotFoundError("Could not find SCP. Set SCP_ROOT or run from the repo.")
    clone_url = SCP_REPO_URL
    token = os.environ.get("SCP_GIT_TOKEN") or os.environ.get("SCP_GITHUB_TOKEN") or os.environ.get("GITHUB_TOKEN")
    if token and clone_url.startswith("https://") and "@" not in clone_url:
        clone_url = clone_url.replace("https://", f"https://{token}@", 1)
    clone_command = ["git", "clone", "--depth", "1"]
    if SCP_REPO_BRANCH:
        clone_command += ["--branch", SCP_REPO_BRANCH]
    clone_command += [clone_url, str(SCP_REPO_DIR)]
    subprocess.check_call(clone_command)
    repo_root = SCP_REPO_DIR.resolve()

os.environ["SCP_ROOT"] = str(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from modules.notebooks.bootstrap import finish_step5_notebook_setup, ensure_python_package
setup = finish_step5_notebook_setup(repo_root, install_deps=INSTALL_DEPS, check_external_inputs=False)
repo_root = setup["repo_root"]
IN_COLAB = setup["in_colab"]
if INSTALL_DEPS is True or (INSTALL_DEPS is None and IN_COLAB):
    ensure_python_package("ipywidgets")

from modules.notebooks import PipelineNotebookUI

## Notebook Setup: Settings

These Python values initialize the panels and remain available for code-first edits. Widget changes update this mapping; rerun this cell to push edited values back into unlocked controls. Model-selection values lock after Step 1 and require a kernel restart to change.

In [ ]:
pipeline_settings = {
    # Step 1: choose an existing tune or optionally prepare an ADB specimen.
    "cell_name": "PV",
    "tune_name": "tuned",
    "tune_dir_override": None,
    "adb_specimen_id": None,
    "adb_model_type": "perisomatic",
    "recompile_modfiles": False,

    # Step 2: proposals are review-only and are never auto-applied.
    "passive_amps_pA": [-50, -100],
    "compute_act_passive_proposal": False,
    "passive_protocol_overrides": {},

    # Step 3: compact active checks and FI curve.
    "active_amps_pA": [150, 300],
    "fi_amps_pA": list(range(0, 301, 50)),
    "active_protocol_overrides": {},

    # Step 4: opt in before initializing BMTool.
    "enable_synapse_tuning": False,
    "synapse_connection": None,

    # Step 5: every final run is saved and loaded for diagnostics.
    "n_trials": 1,
    "seed": None,
    "run_iclamp": False,
    "output_stem": None,
    "diagnostic_plot": "standard",
}

if "pipeline_ui" in globals():
    pipeline_ui.apply_settings(pipeline_settings)

## Step 1: Setup and Load the Tune

Choose or type a cell and tune, then click **Prepare and load**. Optional path and ADB fields support the same compact setup paths as before. A successful load locks model selection because the cell must be constructed only once per kernel.

In [ ]:
if "pipeline_ui" not in globals():
    pipeline_ui = PipelineNotebookUI(repo_root, pipeline_settings)
else:
    pipeline_ui.apply_settings(pipeline_settings)
pipeline_ui.step1_panel()

## Step 2: Passive Tuning

Choose negative current amplitudes and click **Run passive**. The panel reloads `target_config.json` and simulation conditions on every run. ACT proposals require complete Rin, tau, and resting-voltage targets and remain review-only.

In [ ]:
pipeline_ui.step2_panel()

## Step 3: Active Tuning and FI Curve

Choose the positive-current and FI amplitudes, then click **Run active / FI**. Configured FI-array or CSV targets are overlaid and compared automatically.

In [ ]:
pipeline_ui.step3_panel()

## Step 4: Synapse Tuning (Optional)

Enable this optional panel, choose a configured connection, and initialize BMTool around the same shared cell. The compact workflow exposes only **Single Event** and **Interactive Tuner**. Manually copy chosen values into the relevant synapse JSON before Step 5.

In [ ]:
pipeline_ui.step4_panel()

## Step 5: Simulate and Diagnose

Choose the trial count, optional seed, run mode, output name, and diagnostic style, then click **Run simulation**. The run starts in a fresh Python process, reloads current configs, saves its result, and displays diagnostics here.

In [ ]:
pipeline_ui.step5_panel()

## Next: Step 6 Analysis

Open `6_analysis.ipynb` for full saved-run analysis. Return to the corresponding numbered notebook whenever you need the advanced controls intentionally omitted here.